# Strategy Validation

Strategy: **sentiment_synchronous**.

## CPCV Predictions

- **Purpose:** Generate CPCV splits, refit the primary and meta models within each split, and produce test-only action probabilities.
- **Settings:** `num_groups=6`; `num_test_groups=2`; outer/inner `pct_embargo=0.01`; inner `cv=5`; saved estimator settings; weighted fits; composite event keys.
- **Data:** Use prepared development events and frozen model configurations to inspect split-level predictions and action probabilities.
- **Findings:**
- **Decision:** Purge and embargo overlapping information, keep every prediction out of sample within its split, and exclude the holdout partition. Regenerate the saved predictions after changes to data, models, or validation settings.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[2]

from src.backtesting.strategy_validation import (
    assemble_cpcv_path_predictions,
    combinatorial_purged_cross_validation,
    get_cpcv_price_calibrations,
    generate_cpcv_predictions,
)
from src.backtesting.portfolio_management import PortfolioSettings, benchmark_returns, load_event_entry_prices, summarize_account
from src.modeling.model_workflow import (
    build_primary_model_frame,
)
from src.preprocessing.market_data import ResearchPaths, sessions

artifact_dir = PROJECT_ROOT / "data/model_artifact/sentiment"
from src.preprocessing.market_data import load_manifest
PERIOD = "2025-02-01_2025-12-31"
MANIFEST_PATH = PROJECT_ROOT / "data/preprocessing/universe/sp500_2025.csv"
EXPECTED_SECURITIES = 503
DATA_START = pd.Timestamp("2025-01-01", tz="UTC")
END = pd.Timestamp("2026-01-01", tz="UTC")
HORIZON_BARS = 1000
INNER_CV_SPLITS = 5
INNER_PCT_EMBARGO = 0.01
K = 5
STEP_SIZE = 0.10
INITIAL_AUM = 100_000.0
BROKER_FEE_BPS = 1.0
SLIPPAGE_BPS = 1.0
HALF_LIFE_MINUTES = 390.0
PERIODS_PER_YEAR = 365.25
annual_risk_free_rate = 0.03
annualized_benchmark_sharpe_ratio = 1.0
paths = ResearchPaths(PROJECT_ROOT, model_kind="sentiment", period=PERIOD)
universe = load_manifest(MANIFEST_PATH, expected_securities=EXPECTED_SECURITIES)
STRATEGY = "sentiment_synchronous"
settings = PortfolioSettings(
    k=K, step_size=STEP_SIZE, initial_aum=INITIAL_AUM,
    broker_fee_bps=BROKER_FEE_BPS, slippage_bps=SLIPPAGE_BPS,
    half_life_minutes=HALF_LIFE_MINUTES,
)
events = pd.read_parquet(paths.event("model")).sort_values("event_start")
manifest = pd.read_parquet(artifact_dir / "split_manifest.parquet")
development_starts = manifest.loc[manifest["partition"].eq("development"), ["symbol", "event_start"]]
development = build_primary_model_frame(events, development_starts, model_kind="sentiment")

primary_artifact = joblib.load(artifact_dir / "primary_model.joblib")
meta_artifact = joblib.load(artifact_dir / "meta_model.joblib")
information_sets = development["event_end"]

splits = combinatorial_purged_cross_validation(
    samples_info_sets=information_sets,
    num_groups=6,
    num_test_groups=2,
    pct_embargo=0.01,
)
development = load_event_entry_prices(paths, development)
split_calibrations = get_cpcv_price_calibrations(development, splits)

from src.backtesting.portfolio_management import simulate_strategy
prediction_path = artifact_dir / "cpcv_predictions.parquet"

In [ ]:
cpcv_predictions = generate_cpcv_predictions(
    development,
    splits,
    primary_artifact,
    meta_artifact,
    split_calibrations,
    prediction_path=prediction_path,
    inner_cv_splits=INNER_CV_SPLITS,
    inner_pct_embargo=INNER_PCT_EMBARGO,
)
display(cpcv_predictions.head())
display(cpcv_predictions.groupby("split_num")[["meta_probability", "meta_action"]].mean())

## Backtest Paths

- **Purpose:** Evaluate every complete CPCV path through bet sizing, one cross-sectional account, and backtest statistics.
- **Settings:** 5 paths; 1000-bar horizon; split-training calibration; [Shared portfolio settings](portfolio_management.ipynb); annualization = 365.25; risk-free = 3%; benchmark Sharpe = 1.
- **Data:** Assemble each development observation exactly once per path and replay the same market observations for every account.
- **Findings:**
- **Decision:** Keep path accounts and statistics in memory, report their Sharpe distribution, and never use or overwrite the sealed holdout result.

In [ ]:
cpcv_paths = assemble_cpcv_path_predictions(
    cpcv_predictions, splits, development.index, num_groups=6
)
split_summary = splits[["split_num", "train_groups", "test_groups"]].copy()
split_summary["train_events"] = splits["train_indices"].map(len)
split_summary["test_events"] = splits["test_indices"].map(len)

display(split_summary)
display(pd.DataFrame({name: path["meta_probability"].describe() for name, path in cpcv_paths.items()}).T)

calendar = sessions(paths, start=DATA_START, end=END)
holdout_boundary = development["holdout_boundary"].iloc[0]
end = calendar.loc[calendar["close"].lt(holdout_boundary), "close"].max()
from src.backtesting.portfolio_management import load_session_prices
relevant_sessions = calendar.loc[calendar.close.ge(development.index.get_level_values("event_start").min()) & calendar.open.le(end)]
boundary_prices = load_session_prices(paths, relevant_sessions, development.index.get_level_values("symbol").unique())
path_statistics = []
for path_name, path in cpcv_paths.items():
    path_events = path.reset_index().copy()
    path_events["horizon_bars"] = HORIZON_BARS
    path_calibration = path_events[[
        "symbol", "event_start", "w", "calibration_reason"
    ]].rename(columns={"calibration_reason": "reason"})
    start = path_events["event_start"].min()
    account = simulate_strategy(
        paths, path_events, calendar, start, end, STRATEGY, settings,
        path_calibration, evaluation_partition="development", boundary_prices=boundary_prices,
        symbols=universe.symbol.tolist(),
    )
    if account["ledger"].empty:
        raise ValueError(f"{path_name} produced no account ledger")
    account_start = account["ledger"]["timestamp"].min()
    statistics = summarize_account(
        account, settings, benchmark_returns(paths, account_start, end, data_start=DATA_START),
        annual_risk_free_rate=annual_risk_free_rate, periods_per_year=PERIODS_PER_YEAR, annualized_benchmark_sharpe_ratio=annualized_benchmark_sharpe_ratio,
    ).assign(path=path_name)
    path_statistics.append(statistics)

cpcv_statistics = pd.concat(path_statistics, ignore_index=True)
path_sharpe = cpcv_statistics.loc[
    cpcv_statistics["metric"].eq("annualized_sharpe_ratio"),
    ["path", "value"],
].set_index("path")["value"]
display(cpcv_statistics)
display(path_sharpe.rename("annualized_sharpe_ratio").to_frame())
display(path_sharpe.describe().rename("annualized_sharpe_distribution").to_frame())

from src.preprocessing.market_data import save_frame
save_frame(cpcv_statistics, PROJECT_ROOT / "data/backtest_results" / STRATEGY / "cpcv_statistics.parquet")